# Ön İşleme Adımlar

1. Lowercase -> "Merhaba" -> "merhaba"
2. Noktalama kaldır -> "iyi!" -> "iyi"
3. Sayılar(opsiyonel) -> "100 tl" > "tl"
4. Tokenization -> "iyi günler" -> ["iyi", "günler]
5. Stopwords kaldır -> "ve", "bir", "the" temizle
6. Stemming / Lemmatization -> kök bul
7. Normalizasyon -> "gr8" -> "great"

In [1]:
import re

In [ ]:
# Ham Metin
text = """
MERHABA! Bügun NLP öğreniyoruz, değil mi?
Python ile 123 örnek yapacağız: çok eğlenceli :)
E-posta: ornek@code2work.com - telefon: 0555-555-5555
"""

# repr = metni "oldugu gibi gösterir"-> satır sonları \n, boşluklar Görünür olur.
print("Ham:", repr(text))

Ham: '\nMERHABA! Bügun NLP öğreniyoruz, değil mi?\nPython ile 123 örnek yapacağız: çok eğlenceli :)\nE-posta: ornek@code2work.com - telefon: 0555-555-5555\n'


In [3]:
# 1. LOWERCASE
text_lower = text.lower()
print("Lower:", text_lower)

Lower: 
merhaba! bügun nlp öğreniyoruz, değil mi?
python ile 123 örnek yapacağız: çok eğlenceli :)
e-posta: ornek@code2work.com - telefon: 0555-555-5555



In [4]:
# 2. Noktalama KALDIR
# re.sub(kalıp, yeni,metin) -> kaliba uyanları "yeni" ile değiştirir ('', sil)
#    [^...] -> şunları Dışındaki her şey (^ köşeli parante içindeki "değil)"
#     /n -> harf/rakam (word char) * \s -> boşluk (space)   
#     [^\w\s] = harf/rakam/boşluk OLMAYAN -> noktalama, ünlem ,emoji silinir  
text_clean = re.sub(r'[^\w\s]', '', text_lower)
print("No punct:", text_clean)

No punct: 
merhaba bügun nlp öğreniyoruz değil mi
python ile 123 örnek yapacağız çok eğlenceli 
eposta ornekcode2workcom  telefon 05555555555



In [5]:
# 3. SAYILAR (opsiyonel - domaine göre karar)
#    \d -> bir rakam (0-9) * + -> "bir veya daha fazla" 
#.   \d = ardışık rakam grbu (123,4567...) -> sil

text_nonum = re.sub(r"\d+", '', text_clean)
print("No nums:", text_nonum)   

No nums: 
merhaba bügun nlp öğreniyoruz değil mi
python ile  örnek yapacağız çok eğlenceli 
eposta ornekcodeworkcom  telefon 



In [6]:
# 4. FAZLA BOŞLUKLARI TEK BOŞLUĞA
#   \s+ -> bir /daha fazla boşluk/tab/satır sonu -> tek ' ' yap
#   .strip() -> baş/son boşlukları at
text_final = re.sub(r"\s+", ' ', text_nonum).strip()
print("Final:", text_final)

Final: merhaba bügun nlp öğreniyoruz değil mi python ile örnek yapacağız çok eğlenceli eposta ornekcodeworkcom telefon


In [8]:
def temizle(text):
    text = text.lower() # küçük harf
    text = re.sub(r"http\+S|www\S+", '', text) # URL sil (\S+= boşluk olmayan |= "veya")
    text = re.sub(r"\S+@\S+", '', text) # e-posta sil (xxx@yyy)
    text = re.sub(r"[^\w\sğüşıöĞÜŞİÖÇ]", '', text) # noktalama sil - Türkçe harf KORU
    text = re.sub(r"\d+", '', text) # sayı sil
    text = re.sub(r"\s+", ' ', text).strip()
    return text

sample = "Bugün saat 14:40'da https://code2work.co sitesini ziyaret ettim. Harikaydı!"
print(temizle(sample))

bugün saat da httpscodeworkco sitesini ziyaret ettim harikaydı


In [9]:
# STOPWORDS - anlam taşımayan kelimeler
stopwords_tr = ["ve", "bir", "bu", "için", "ile", "de", "da", "mi", "mu",
             "mü", "gibi", "çok", "ama", "en", "daha", "ne", "ki", "her"]
stopword_en = ["the", "a", "an", "and", "or", "but", "in", "on", "at", "to","for"
               "of", "with", "is", "are", "was", "were", "be", "been", "being"]

def remove_stopwords(text, stops):
    words = text.split() # Cümleyi kelimelere böl
    kept = [] # stopwords OLAMYAN kelimeleri biriktireceğiz.

    for w in words:
        if w not in stops: # kelime stopword listesinde DEĞİLSE
            kept.append(w) # listeye ekle (anlamlı kelime)

    return ' '.join(kept) # kalan kelimeleri tekrar cümleye çevir.

cumle = "Bu bir çok güzel bir örnek ve anlamsızdır"
print("HAM:, cumle")
print("No SW:", remove_stopwords(cumle, stopwords_tr))

HAM:, cumle
No SW: Bu güzel örnek anlamsızdır


In [11]:
# NLTK stopwords listesi (daha geniş)
try:
    import nltk
    nltk.download('stopwords', quiet= True)
    from nltk.corpus import stopwords

    sw_en = set(stopwords.words("english"))
    sw_tr = set(stopwords.words("turkish"))

    print(f"İngilizce stopwords: {len(sw_en)}")
    print(f"Türkçe stopwords: {len(sw_tr)}")
    print(f"Örnek: {list(sw_tr)[:10]}")

except ImportError:
    print("pip install nltk")

İngilizce stopwords: 198
Türkçe stopwords: 53
Örnek: ['daha', 'hiç', 'çünkü', 'da', 'nasıl', 'o', 'ile', 'belki', 'de', 'niçin']


## Dikkat Noktaları

1. Sentiment için: "değil", "ama" -> silmeme! Anlam bozulur
2. Domain'e göre: medikal -> "kanser" stopword olamaz
3. Türkçe özel: "ğ","ş","ü","ç" - encoding dikkat
4. Emojis: sentiment için değerli, silmek yanlış olabilir
5. URL/email: domain'e göre silme veya placeholder